In [2]:
from pathlib import Path
import cv2

def load_faces(root):
    root = Path(root)
    data = {}  # {person_id: [img1, img2, ...]}
    
    for person_dir in sorted(root.glob("s*")):
        pid = int(person_dir.name[1:])
        imgs = []
        for img_path in person_dir.glob("*.pgm"):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            imgs.append(img)
        data[pid] = imgs
    
    return data

faces_by_id = load_faces("/home/bargomix/Documents/secondcourse/PAC/semister4/lab2/datasets/kasikrit/att-database-of-faces/versions/2")
print(len(faces_by_id), "people loaded")

40 people loaded


In [3]:
import random

def make_pairs(data, n_pairs=20000):
    ids = list(data.keys())
    pairs = []
    
    # 50% одинаковые
    for _ in range(n_pairs // 2):
        pid = random.choice(ids)
        a, b = random.sample(data[pid], 2)
        pairs.append((a, b, 1))
    
    # 50% разные
    for _ in range(n_pairs // 2):
        pid1, pid2 = random.sample(ids, 2)
        a = random.choice(data[pid1])
        b = random.choice(data[pid2])
        pairs.append((a, b, 0))
    
    random.shuffle(pairs)
    return pairs

pairs = make_pairs(faces_by_id, n_pairs=20000)

In [4]:
def split_pairs(pairs, val_ratio=0.2):
    n = len(pairs)
    val_n = int(n * val_ratio)
    return pairs[val_n:], pairs[:val_n]

train_pairs, val_pairs = split_pairs(pairs)

In [5]:
import torchvision.transforms as T
from PIL import Image

class ToPIL:
    def __call__(self, img):
        return Image.fromarray(img)

transform = T.Compose([
    ToPIL(),
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class FaceDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1, img2, label = self.pairs[idx]
        return (
            self.transform(img1),
            self.transform(img2),
            torch.tensor(label, dtype=torch.float32)
        )

train_ds = FaceDataset(train_pairs, transform)
val_ds   = FaceDataset(val_pairs, transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

In [7]:
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class EmbeddingNet(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        m = models.resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        m.fc = nn.Linear(m.fc.in_features, emb_dim)
        self.net = m

    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = EmbeddingNet()

    def forward(self, x1, x2):
        return self.emb(x1), self.emb(x2)

In [8]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, y):
        d = F.pairwise_distance(z1, z2)
        loss = y * d**2 + (1 - y) * torch.clamp(self.margin - d, min=0)**2
        return loss.mean(), d

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model = SiameseNet().to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(loader):
    model.train()
    total = 0
    for x1, x2, y in loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad()
        z1, z2 = model(x1, x2)
        loss, _ = criterion(z1, z2, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * x1.size(0)
    return total / len(loader.dataset)

def eval_epoch(loader):
    model.eval()
    dists, labels = [], []
    with torch.no_grad():
        for x1, x2, y in loader:
            x1, x2 = x1.to(device), x2.to(device)
            z1, z2 = model(x1, x2)
            d = F.pairwise_distance(z1, z2)
            dists.append(d.cpu())
            labels.append(y)
    return torch.cat(dists), torch.cat(labels)

cuda


In [10]:
for epoch in range(10):
    loss = train_epoch(train_loader)
    print(f"Epoch {epoch+1}: loss={loss:.4f}")

Epoch 1: loss=0.0443
Epoch 2: loss=0.0159
Epoch 3: loss=0.0100
Epoch 4: loss=0.0068
Epoch 5: loss=0.0042
Epoch 6: loss=0.0029
Epoch 7: loss=0.0023
Epoch 8: loss=0.0015
Epoch 9: loss=0.0008
Epoch 10: loss=0.0006


In [11]:
dists, labels = eval_epoch(val_loader)

# простой порог
threshold = dists.mean().item()

pred = (dists < threshold).float()
acc = (pred == labels).float().mean()
print("Validation accuracy:", acc.item())

Validation accuracy: 1.0


In [12]:
import torch

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 2060
